In [6]:
# !pip install segmentation_models
# !pip install keras.api
import os
os.environ["SM_FRAMEWORK"] = "tf.keras"
import tensorflow as tf
import segmentation_models as sm
import glob
import cv2
import os
import numpy as np
from matplotlib import pyplot as plt
import keras 

from keras.utils import normalize
from keras.metrics import MeanIoU


#Resizing images, if needed
SIZE_X = 128 
SIZE_Y = 128
n_classes=3 #Number of classes for segmentation

#Capture training image info as a list
train_images = []

for directory_path in glob.glob("augmented_images"):
    for img_path in glob.glob(os.path.join(directory_path, "*.jpg")):
        # if(img_path ends
        img = cv2.imread(img_path, 1)       
        img = cv2.resize(img, (SIZE_Y, SIZE_X))
        train_images.append(img)

train_images= np.array(train_images)
print(train_images.shape)

(696, 128, 128, 3)


In [7]:
train_masks = [] 
for directory_path in glob.glob("augmented_masks"):
    for mask_path in glob.glob(os.path.join(directory_path, "*jpg")):
        
        mask = cv2.imread(mask_path, 0)     
        mask = cv2.resize(mask, (SIZE_Y, SIZE_X), interpolation = cv2.INTER_NEAREST)  #Otherwise ground truth changes due to interpolation
        train_masks.append(mask)
        
#Convert list to array for machine learning processing          
train_masks = np.array(train_masks)
print(train_masks.shape)
for i in train_masks[0]:
    for j in i:
        if j>0:
            print(j)


(696, 128, 128)
8
9
2
3
4
15
1
7
3
3
12
7
4
9
6
9
15
2
7
12
238
255
253
12
3
11
6
6
9
5
19
1
6
252
255
255
255
251
255
2
10
7
3
6
10
25
3
7
241
255
255
254
255
255
249
255
11
5
11
10
5
10
7
249
255
232
255
255
247
251
254
9
2
2
2
20
14
4
2
5
255
238
255
255
237
255
241
255
251
5
11
11
9
5
22
21
251
255
255
250
255
255
245
255
236
252
254
255
12
8
12
9
5
6
4
251
255
255
255
247
255
255
255
255
255
253
3
1
4
252
242
255
255
249
250
255
253
255
255
255
9
2
4
7
255
255
223
255
251
254
255
255
237
255
7
6
9
248
254
255
252
254
249
252
255
255
2
3
21
26
241
243
255
255
255
251
250
254
252
4
4
10
8
255
254
240
246
237
255
250
255
13
6
5
255
255
255
255
255
255
254
255
17
1
1
8
250
254
255
255
242
252
235
16
4
4
1
255
6
20
5


In [3]:
# !pip install sklearn
np.array(train_masks)
from sklearn.preprocessing import LabelEncoder
labelencoder = LabelEncoder()

n, h, w = train_masks.shape
train_masks_reshaped = train_masks.reshape(-1,1)
train_masks_reshaped_encoded = labelencoder.fit_transform(train_masks_reshaped)
train_masks_encoded_original_shape = train_masks_reshaped_encoded.reshape(n, h, w)
np.unique(train_masks_encoded_original_shape)

c:\Python312\Lib\site-packages\sklearn\preprocessing\_label.py:114: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
        26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
        39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
        52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
        65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
        78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
        91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
       104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116,
       117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129,
       130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142,
       143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155,
       156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168,
       169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 18